# [C4 / B3 boundary] STTR vs. SBIR award-record timing readiness

**Status:** exploratory, non-citable  
**Research question:** [C4 — STTR award-record timing and administrative friction](../../docs/research-questions.md#c4-sttr-award-timing-friction)  
**Decision this informs:** whether public award records are ready for a descriptive, within-stratum STTR/SBIR timing comparison, and which source strata need a data-quality or primary-data follow-up. It does not decide whether STTR creates more administrative friction.  
**Data as of:** reference audit uses the SBIR.gov bulk CSV retrieved 2026-10-21; the executed input, SHA-256, and effective as-of date are reported below.  
**Owner:** exploratory workbench

Question: within observed agency × solicitation × phase × solicitation-year strata, where do source-labeled award-record intervals differ between STTR and SBIR, and where is coverage sufficient to compare them?

This notebook refuses a naive pooled faster/slower result. It keeps Phase I and Phase II separate, requires common strata with declared interval coverage, and standardizes only over those observed common strata. B3 Phase II→III latency starts after Phase II completion; this notebook examines dates recorded on public source-key records and does not estimate that transition clock.


## Data contract

- **Population:** raw CSV rows in one public SBIR.gov export whose labels are exactly `SBIR` or `STTR` and whose phase is exactly Phase I or Phase II. Direct-to-Phase-II rows have no STTR counterpart here and are excluded from comparisons. The export contains awardees only, never unsuccessful applicants.
- **Grain:** one `sbir-source-v2` key from the repository identity primitive. A key is a governed source-record identity, **not a verified physical award**. Repeated editions collapse only after interval-specific conflict flags are retained.
- **Keys:** source record = `award_key`; comparison stratum = `Agency` × `Solicitation Number` × `Phase` × `Solicitation Year`. A public ID alone is not assumed unique. Plausible split-key lineage groups are audited and never collapsed without a governed rule.
- **Inputs:** set `SBIR_AWARDS_CSV` to an exact SBIR.gov bulk CSV. Otherwise the notebook checks `data/raw/sbir/award_data.csv`. Set `SBIR_AWARDS_AS_OF` when the snapshot is not the reference hash. No download occurs. A present file missing any required identity, stratum, or endpoint column produces a refusal; required columns are never silently filled.
- **Date rule:** use only present source columns under their literal labels. The repository has an ordered-column schema but no field-level business-rule dictionary for these dates. The intervals are therefore **record intervals**, not verified workflow clocks.
- **Plausibility:** each interval must be 0–730 days inclusive. Missing, unparseable, negative, and over-730-day records remain visible in the audit and are excluded from contrasts.
- **Denominator:** for each program × interval × stratum, count every stable-stratum source key before endpoint-quality filtering. Endpoint conflicts, missing dates, and implausible intervals remain in the denominator.
- **Common support:** the primary readiness rule requires at least five plausible source-key intervals **and at least 80% plausible-interval coverage** for each program in the same stratum. Both cutoffs are conservative readiness thresholds, not statistical truths or power claims. Count-only versus 80% sensitivity is mandatory.
- **Standardization:** report equal-stratum and symmetric overlap-weighted mean contrasts by phase. No pooled median is computed. The target is the observed supported strata, not either full program.
- **Edition conflicts:** stratum conflicts exclude a key from every comparison. Endpoint conflicts exclude that key only for intervals using the disputed endpoint. A `Contract End Date` revision therefore cannot suppress a stable receipt-to-notification observation.
- **Exclusions:** missing or conflicting stratum fields, unsupported cells, interval-specific endpoint conflicts, Direct-to-Phase-II, and programs/phases outside the declared population.
- **Assumptions:** matching source labels mean comparable recorded events; shared solicitation IDs are only grouping keys and can span topics, deadlines, panels, or review paths.
- **Seed:** `20261028`; no stochastic estimator is used.
- **Outputs:** in-notebook readiness, quality, stratum, and standardization tables only. No canonical generator and no citable result.


In [ ]:
from __future__ import annotations

import hashlib
import os
from pathlib import Path

import pandas as pd

from sbir_etl.identity.sbir_awards import (
    SBIR_AWARD_KEY_FIELDS,
    SBIR_AWARD_KEY_VERSION,
    sbir_award_grain_key,
)


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT


In [ ]:
INPUT_ENV = "SBIR_AWARDS_CSV"
AS_OF_ENV = "SBIR_AWARDS_AS_OF"
REFERENCE_AS_OF_DATE = "2026-10-21"
REFERENCE_SHA256 = "1058456da366796402ad5264bbd90cfcd0e1de6dc8206db372579e06de27c600"
SOURCE_URL = "https://data.www.sbir.gov/mod_awarddatapublic/award_data.csv"
DEFAULT_AWARDS_PATH = REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv"
AWARDS_PATH = Path(os.environ.get(INPUT_ENV, DEFAULT_AWARDS_PATH)).expanduser()
RANDOM_SEED = 20261028
MAX_INTERVAL_DAYS = 730
MIN_PROGRAM_ROWS = 5
MIN_INTERVAL_COVERAGE = 0.80
COVERAGE_THRESHOLDS = (0.0, MIN_INTERVAL_COVERAGE)
MIN_SOLICITATION_YEAR = 1982
MAX_SOLICITATION_YEAR = int(os.environ.get("SBIR_MAX_SOLICITATION_YEAR", "2026"))
PROGRAMS = ("SBIR", "STTR")
PHASES = ("PHASE I", "PHASE II")

DATE_FIELDS = {
    "Solicitation Close Date": "solicitation_close_date",
    "Proposal Receipt Date": "proposal_receipt_date",
    "Date of Notification": "notification_date",
    "Proposal Award Date": "proposal_award_date",
    "Contract End Date": "contract_end_date",
}
KEY_SOURCE_COLUMNS = tuple(names[0] for names in SBIR_AWARD_KEY_FIELDS)
IDENTITY_COLUMNS = ("Company", "UEI", "Duns", *KEY_SOURCE_COLUMNS)
STRATUM_SOURCE_COLUMNS = ("Agency", "Solicitation Number", "Phase", "Solicitation Year")
REQUIRED_SOURCE_COLUMNS = tuple(
    dict.fromkeys((*IDENTITY_COLUMNS, *STRATUM_SOURCE_COLUMNS, *DATE_FIELDS))
)
OPTIONAL_SOURCE_COLUMNS = ("Topic Code",)
ANALYSIS_COLUMNS = tuple(
    dict.fromkeys(
        (*REQUIRED_SOURCE_COLUMNS, *OPTIONAL_SOURCE_COLUMNS)
    )
)
INTERVAL_SPECS = {
    "receipt_to_notification": {
        "start_source": "Proposal Receipt Date",
        "start": "proposal_receipt_date",
        "end_source": "Date of Notification",
        "end": "notification_date",
        "role": "primary comparison",
        "compare": True,
    },
    "receipt_to_award": {
        "start_source": "Proposal Receipt Date",
        "start": "proposal_receipt_date",
        "end_source": "Proposal Award Date",
        "end": "proposal_award_date",
        "role": "secondary comparison",
        "compare": True,
    },
    "close_to_notification": {
        "start_source": "Solicitation Close Date",
        "start": "solicitation_close_date",
        "end_source": "Date of Notification",
        "end": "notification_date",
        "role": "secondary comparison",
        "compare": True,
    },
    "notification_to_award": {
        "start_source": "Date of Notification",
        "start": "notification_date",
        "end_source": "Proposal Award Date",
        "end": "proposal_award_date",
        "role": "secondary comparison",
        "compare": True,
    },
    "award_to_contract_end": {
        "start_source": "Proposal Award Date",
        "start": "proposal_award_date",
        "end_source": "Contract End Date",
        "end": "contract_end_date",
        "role": "quality audit only",
        "compare": False,
    },
}


In [ ]:
def file_sha256(path: Path) -> str | None:
    if not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_sha256 = file_sha256(AWARDS_PATH)
effective_as_of = os.environ.get(AS_OF_ENV)
if not effective_as_of and source_sha256 == REFERENCE_SHA256:
    effective_as_of = REFERENCE_AS_OF_DATE
effective_as_of = effective_as_of or "unknown — set SBIR_AWARDS_AS_OF"
input_status = pd.DataFrame(
    [
        {
            "input": "SBIR.gov bulk award CSV",
            "path": str(AWARDS_PATH),
            "exists": AWARDS_PATH.is_file(),
            "bytes": AWARDS_PATH.stat().st_size if AWARDS_PATH.is_file() else pd.NA,
            "sha256": source_sha256,
            "as_of": effective_as_of,
            "reference_hash_match": source_sha256 == REFERENCE_SHA256,
            "source_url": SOURCE_URL,
            "override": INPUT_ENV,
        }
    ]
)
input_status


## Field semantics and refusal rules

The exact 42-column repository schema establishes that these labels exist; it does not document how agencies populate them. This notebook mirrors the labels and does not invent a `start_date`.

| Source label | Use here | What it cannot establish |
|---|---|---|
| `Proposal Receipt Date` | start of two record intervals | first submission, resubmission, completeness, or review start |
| `Date of Notification` | end of the primary record interval | decision date, panel date, or uniform notice process |
| `Proposal Award Date` | end of a sensitivity interval | contract start or obligation date |
| `Solicitation Close Date` | start of a secondary interval | the applicable deadline for every award under a multi-deadline solicitation |
| `Contract End Date` | audit-only performance-span endpoint | administrative selection timing |
| `Contract` | identifier only | any date |
| start date | absent from the export | award-to-start timing |

The primary diagnostic is `Proposal Receipt Date`→`Date of Notification`. Secondary intervals test whether the readiness verdict depends on the chosen source labels. `Proposal Award Date`→`Contract End Date` is audited for missing and implausible values but never used as administrative timing.


In [ ]:
def clean_text(values: pd.Series) -> pd.Series:
    return values.fillna("").astype(str).str.strip()


def normalize_token(values: pd.Series) -> pd.Series:
    return clean_text(values).str.replace(r"\s+", " ", regex=True).str.upper()


def parse_mixed_dates(values: pd.Series) -> pd.Series:
    return pd.to_datetime(clean_text(values).replace("", pd.NA), errors="coerce", format="mixed")


def date_conflict_token(raw: pd.Series, parsed: pd.Series) -> pd.Series:
    token = normalize_token(raw).replace("", "<MISSING>").astype("string")
    token.loc[parsed.notna()] = parsed.loc[parsed.notna()].dt.strftime("%Y-%m-%d")
    return token


def read_award_rows(path: Path) -> tuple[pd.DataFrame, list[str], str, list[str]]:
    if not path.is_file():
        print(f"Missing award CSV. Set {INPUT_ENV} or place the file at {DEFAULT_AWARDS_PATH}.")
        return pd.DataFrame(columns=ANALYSIS_COLUMNS), [], "missing", []

    try:
        header = list(pd.read_csv(path, nrows=0).columns)
    except pd.errors.EmptyDataError:
        print("Award CSV is present but contains no parseable columns or rows.")
        return pd.DataFrame(columns=ANALYSIS_COLUMNS), [], "empty", []
    missing_required = [column for column in REQUIRED_SOURCE_COLUMNS if column not in header]
    if missing_required:
        print("Malformed award CSV; required columns are missing: " + ", ".join(missing_required))
        return pd.DataFrame(columns=ANALYSIS_COLUMNS), header, "malformed", missing_required

    available = [column for column in ANALYSIS_COLUMNS if column in header]
    frame = pd.read_csv(
        path,
        usecols=available,
        dtype=str,
        keep_default_na=False,
        low_memory=False,
    )
    if frame.empty:
        print("Award CSV has the required schema but no source rows.")
        return pd.DataFrame(columns=ANALYSIS_COLUMNS), header, "empty", []
    for column in OPTIONAL_SOURCE_COLUMNS:
        if column not in frame:
            frame[column] = ""
    frame = frame.loc[:, list(ANALYSIS_COLUMNS)].copy()
    frame.insert(0, "source_row", range(2, len(frame) + 2))
    frame["program_n"] = normalize_token(frame["Program"])
    frame["phase_n"] = normalize_token(frame["Phase"])
    frame["agency_n"] = clean_text(frame["Agency"])
    frame["agency_token"] = normalize_token(frame["Agency"])
    frame["solicitation_n"] = normalize_token(frame["Solicitation Number"])
    frame["topic_n"] = normalize_token(frame["Topic Code"])
    solicitation_year_raw = normalize_token(frame["Solicitation Year"])
    frame["solicitation_year"] = pd.to_numeric(
        solicitation_year_raw, errors="coerce"
    ).astype("Int64")
    frame["solicitation_year_token"] = solicitation_year_raw.replace("", "<MISSING>")
    parsed_year = frame["solicitation_year"].notna()
    frame.loc[parsed_year, "solicitation_year_token"] = (
        frame.loc[parsed_year, "solicitation_year"].astype(str)
    )
    for source_field, parsed_field in DATE_FIELDS.items():
        frame[parsed_field] = parse_mixed_dates(frame[source_field])
        frame[f"{parsed_field}_token"] = date_conflict_token(
            frame[source_field], frame[parsed_field]
        )
    return frame, header, "valid", []


In [ ]:
source_rows, source_columns, source_state, missing_required_columns = read_award_rows(AWARDS_PATH)
schema_validation = pd.DataFrame(
    [
        {
            "state": source_state,
            "required_columns": len(REQUIRED_SOURCE_COLUMNS),
            "missing_required_columns": ", ".join(missing_required_columns),
            "action": "analyze" if source_state == "valid" else "refuse",
        }
    ]
)
scope_raw = pd.DataFrame()
awards = pd.DataFrame(columns=ANALYSIS_COLUMNS)
interval_conflict_audit = pd.DataFrame()
split_key_lineage_candidates = pd.DataFrame()
lineage_audit = pd.DataFrame(
    [{"metric": "plausible split-key lineage groups", "value": 0, "action": "do not collapse"}]
)
if source_state != "valid":
    edition_audit = pd.DataFrame(
        [{"metric": "retained source keys", "value": 0, "note": f"input state: {source_state}"}]
    )
else:
    in_scope = source_rows["program_n"].isin(PROGRAMS) & source_rows["phase_n"].isin(PHASES)
    direct_phase_ii = source_rows["phase_n"].str.contains("DIRECT", na=False)
    scope_raw = source_rows.loc[in_scope].copy()
    if not scope_raw.empty:
        scope_raw["award_key"] = scope_raw.apply(sbir_award_grain_key, axis=1)
        grouped = scope_raw.groupby("award_key", sort=False, dropna=False)
        edition_size = grouped.size()
        stratum_tokens = [
            "program_n", "agency_token", "solicitation_n", "phase_n",
            "solicitation_year_token",
        ]
        stratum_conflict = grouped[stratum_tokens].nunique(dropna=False).gt(1).any(axis=1)
        endpoint_conflicts = {}
        for interval, spec in INTERVAL_SPECS.items():
            endpoint_tokens = [f"{spec['start']}_token", f"{spec['end']}_token"]
            endpoint_conflicts[interval] = (
                grouped[endpoint_tokens].nunique(dropna=False).gt(1).any(axis=1)
            )
        awards = (
            scope_raw.sort_values("source_row", kind="stable")
            .drop_duplicates("award_key", keep="last")
            .copy()
        )
        awards["edition_rows"] = awards["award_key"].map(edition_size).astype(int)
        awards["stratum_conflict"] = awards["award_key"].map(stratum_conflict).fillna(False)
        conflict_rows = []
        for interval, conflicts in endpoint_conflicts.items():
            flag = f"{interval}_endpoint_conflict"
            awards[flag] = awards["award_key"].map(conflicts).fillna(False)
            both = stratum_conflict & conflicts
            conflict_rows.append(
                {
                    "interval": interval,
                    "stratum_conflict_keys": int(stratum_conflict.sum()),
                    "endpoint_conflict_keys": int(conflicts.sum()),
                    "both_conflict_keys": int(both.sum()),
                    "stable_for_interval_keys": int((~(stratum_conflict | conflicts)).sum()),
                }
            )
        interval_conflict_audit = pd.DataFrame(conflict_rows)

        # Audit exact non-recipient key tuples that split only when the recipient token changes.
        # These are review candidates, not governed merges.
        lineage = awards.copy()
        lineage_tokens = []
        for column in KEY_SOURCE_COLUMNS:
            token_column = f"_lineage_{column}"
            if column == "Proposal Award Date":
                lineage[token_column] = lineage["proposal_award_date_token"]
            else:
                lineage[token_column] = normalize_token(lineage[column]).replace("", "<MISSING>")
            lineage_tokens.append(token_column)
        uei_token = normalize_token(lineage["UEI"])
        duns_token = normalize_token(lineage["Duns"])
        company_token = normalize_token(lineage["Company"])
        lineage["recipient_token"] = company_token.replace("", "<MISSING>")
        lineage["recipient_token_source"] = "COMPANY"
        lineage.loc[duns_token.ne(""), "recipient_token"] = duns_token[duns_token.ne("")]
        lineage.loc[duns_token.ne(""), "recipient_token_source"] = "DUNS"
        lineage.loc[uei_token.ne(""), "recipient_token"] = uei_token[uei_token.ne("")]
        lineage.loc[uei_token.ne(""), "recipient_token_source"] = "UEI"
        lineage_material = lineage[lineage_tokens].astype(str).agg(chr(31).join, axis=1)
        lineage["lineage_group_id"] = lineage_material.map(
            lambda value: hashlib.sha256(value.encode()).hexdigest()[:16]
        )
        public_identifier_present = (
            normalize_token(lineage["Agency Tracking Number"]).ne("")
            | normalize_token(lineage["Contract"]).ne("")
        )
        lineage["public_identifier_present"] = public_identifier_present
        lineage_groups = lineage.groupby("lineage_group_id", dropna=False).agg(
            source_keys=("award_key", "nunique"),
            recipient_tokens=("recipient_token", "nunique"),
            public_identifier_present=("public_identifier_present", "max"),
        )
        candidate_groups = lineage_groups.loc[
            lineage_groups["source_keys"].gt(1)
            & lineage_groups["recipient_tokens"].gt(1)
            & lineage_groups["public_identifier_present"]
        ].copy()
        split_key_lineage_candidates = (
            lineage.loc[lineage["lineage_group_id"].isin(candidate_groups.index), [
                "lineage_group_id", "award_key", "recipient_token_source",
                "recipient_token", "Company", "UEI", "Duns",
                "Agency Tracking Number", "Contract", "Agency", "Phase", "Program",
            ]]
            .sort_values(["lineage_group_id", "award_key"])
        )
        lineage_audit = pd.DataFrame(
            [
                {
                    "metric": "plausible split-key lineage groups",
                    "value": len(candidate_groups),
                    "action": "audit only; do not collapse",
                },
                {
                    "metric": "source keys in those groups",
                    "value": int(candidate_groups["source_keys"].sum()),
                    "action": "possible recipient-identifier corrections or distinct records",
                },
            ]
        )
        edition_audit = pd.DataFrame(
            [
                {"metric": "raw CSV rows", "value": len(source_rows), "note": "all programs/phases"},
                {"metric": "in-scope source rows", "value": len(scope_raw), "note": "SBIR/STTR; Phase I/II"},
                {"metric": "Direct-to-Phase-II rows excluded", "value": int(direct_phase_ii.sum()), "note": "no STTR counterpart"},
                {"metric": "distinct source keys", "value": int(edition_size.size), "note": SBIR_AWARD_KEY_VERSION},
                {"metric": "keys with repeated editions", "value": int(edition_size.gt(1).sum()), "note": "conflicts retained by interval"},
                {"metric": "keys with stratum conflicts", "value": int(stratum_conflict.sum()), "note": "excluded from all comparisons"},
                {"metric": "retained source keys", "value": len(awards), "note": "not verified physical awards"},
            ]
        )
    else:
        edition_audit = pd.DataFrame(
            [{"metric": "retained source keys", "value": 0, "note": "no in-scope rows"}]
        )
display(schema_validation)
display(edition_audit)
display(interval_conflict_audit)
display(lineage_audit)
split_key_lineage_candidates


In [ ]:
field_rows = []
for source_field, parsed_field in DATE_FIELDS.items():
    raw_values = clean_text(awards[source_field]) if source_field in awards else pd.Series(dtype=str)
    parsed = awards[parsed_field] if parsed_field in awards else pd.Series(dtype="datetime64[ns]")
    nonblank = raw_values.ne("")
    parseable = parsed.notna()
    field_rows.append(
        {
            "source_field": source_field,
            "column_present": source_field in source_columns,
            "source_keys": len(awards),
            "nonblank": int(nonblank.sum()),
            "parseable": int(parseable.sum()),
            "nonblank_unparseable": int((nonblank & ~parseable).sum()),
            "source_key_coverage": float(parseable.mean()) if len(awards) else pd.NA,
            "definition_status": "source label only; no field-level business-rule dictionary",
        }
    )
date_field_readiness = pd.DataFrame(field_rows)

stratum_field_readiness = pd.DataFrame(
    [
        {"field": "Agency", "nonblank": int(awards.get("agency_n", pd.Series(dtype=str)).ne("").sum())},
        {"field": "Solicitation Number", "nonblank": int(awards.get("solicitation_n", pd.Series(dtype=str)).ne("").sum())},
        {"field": "Phase", "nonblank": int(awards.get("phase_n", pd.Series(dtype=str)).ne("").sum())},
        {"field": "Solicitation Year", "nonblank": int(awards.get("solicitation_year", pd.Series(dtype="Int64")).notna().sum())},
        {"field": "Topic Code (limitation audit only)", "nonblank": int(awards.get("topic_n", pd.Series(dtype=str)).ne("").sum())},
    ]
).assign(source_keys=len(awards))
stratum_field_readiness["coverage"] = (
    stratum_field_readiness["nonblank"] / len(awards) if len(awards) else pd.NA
)
display(date_field_readiness)
stratum_field_readiness


## Interval-quality audit

Every interval remains separate. A missing, conflicting, or implausible interval is a data-quality state, not evidence of fast or slow administration. The common 0–730-day gate is deliberately conservative and is applied before any program comparison. Edition conflicts are interval-specific: a revised `Contract End Date` affects only the audit interval. The award-to-contract-end interval is audit-only because performance duration is not selection friction and valid Phase II work can exceed two years.


In [ ]:
def build_interval_rows(frame: pd.DataFrame) -> pd.DataFrame:
    base_columns = [
        "award_key", "program_n", "phase_n", "agency_n",
        "solicitation_n", "solicitation_year", "topic_n",
    ]
    pieces = []
    for interval, spec in INTERVAL_SPECS.items():
        piece = frame.reindex(columns=base_columns).copy()
        start_raw = clean_text(frame.get(spec["start_source"], pd.Series("", index=frame.index)))
        end_raw = clean_text(frame.get(spec["end_source"], pd.Series("", index=frame.index)))
        start = pd.to_datetime(
            frame.get(spec["start"], pd.Series(pd.NaT, index=frame.index)), errors="coerce"
        )
        end = pd.to_datetime(
            frame.get(spec["end"], pd.Series(pd.NaT, index=frame.index)), errors="coerce"
        )
        days = (end - start).dt.days.astype("Int64")
        status = pd.Series("plausible", index=frame.index, dtype="string")
        status.loc[start_raw.eq("") | end_raw.eq("")] = "missing_endpoint"
        status.loc[(start_raw.ne("") & start.isna()) | (end_raw.ne("") & end.isna())] = (
            "unparseable_endpoint"
        )
        status.loc[days.lt(0).fillna(False)] = "negative"
        status.loc[days.gt(MAX_INTERVAL_DAYS).fillna(False)] = f"over_{MAX_INTERVAL_DAYS}"
        stratum_conflict = frame.get(
            "stratum_conflict", pd.Series(False, index=frame.index, dtype=bool)
        ).fillna(False).astype(bool)
        endpoint_conflict = frame.get(
            f"{interval}_endpoint_conflict", pd.Series(False, index=frame.index, dtype=bool)
        ).fillna(False).astype(bool)
        conflict_state = pd.Series("stable", index=frame.index, dtype="string")
        conflict_state.loc[stratum_conflict & ~endpoint_conflict] = "stratum_conflict"
        conflict_state.loc[~stratum_conflict & endpoint_conflict] = "endpoint_conflict"
        conflict_state.loc[stratum_conflict & endpoint_conflict] = (
            "stratum_and_endpoint_conflict"
        )
        status.loc[stratum_conflict & ~endpoint_conflict] = "conflicting_stratum"
        status.loc[~stratum_conflict & endpoint_conflict] = "conflicting_endpoint"
        status.loc[stratum_conflict & endpoint_conflict] = (
            "conflicting_stratum_and_endpoint"
        )
        piece["interval"] = interval
        piece["role"] = spec["role"]
        piece["days"] = days.to_numpy()
        piece["status"] = status.to_numpy()
        piece["conflict_state"] = conflict_state.to_numpy()
        piece["comparison_interval"] = spec["compare"]
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


interval_rows = build_interval_rows(awards)
if interval_rows.empty:
    interval_quality = pd.DataFrame()
else:
    interval_quality = (
        interval_rows.groupby(
            ["interval", "role", "program_n", "phase_n", "conflict_state", "status"],
            dropna=False,
        )
        .size()
        .rename("source_keys")
        .reset_index()
    )
interval_quality


## Common-support strata

The comparison never treats all SBIR source keys as the counterfactual for all STTR source keys. For each program and stratum, the denominator contains every source key with stable, complete stratum fields. Missing dates, endpoint conflicts, and implausible intervals remain in that denominator.

The primary readiness rule requires both programs to have at least `MIN_PROGRAM_ROWS = 5` plausible intervals and `MIN_INTERVAL_COVERAGE = 0.80` plausible intervals divided by denominator keys. This is a conservative readiness threshold, not a statistical truth or power calculation. Count-only and 80% results are shown side by side.

This controls the observed agency, solicitation identifier, phase, and solicitation-year mix. It does **not** control topic, firm, institute partner, deadline, resubmission path, panel, review criteria, or selection rule. A shared solicitation identifier is not proof that two proposals faced the same administrative process.


In [ ]:
STRATUM_COLUMNS = ["agency_n", "solicitation_n", "phase_n", "solicitation_year"]
comparison_universe = interval_rows.loc[
    interval_rows.get("comparison_interval", pd.Series(dtype=bool)).fillna(False)
].copy()
if not comparison_universe.empty:
    comparison_universe["complete_stratum"] = (
        comparison_universe["agency_n"].ne("")
        & comparison_universe["solicitation_n"].ne("")
        & comparison_universe["phase_n"].isin(PHASES)
        & comparison_universe["solicitation_year"].between(
            MIN_SOLICITATION_YEAR, MAX_SOLICITATION_YEAR, inclusive="both"
        )
    )
    comparison_universe["stable_stratum"] = ~comparison_universe[
        "conflict_state"
    ].isin({"stratum_conflict", "stratum_and_endpoint_conflict"})
else:
    comparison_universe["complete_stratum"] = pd.Series(dtype=bool)
    comparison_universe["stable_stratum"] = pd.Series(dtype=bool)
denominator_rows = comparison_universe.loc[
    comparison_universe["complete_stratum"] & comparison_universe["stable_stratum"]
].copy()


def build_stratum_comparisons(frame: pd.DataFrame) -> pd.DataFrame:
    index = ["interval", *STRATUM_COLUMNS]
    columns = [
        *index, "denominator_sbir", "denominator_sttr",
        "plausible_n_sbir", "plausible_n_sttr",
        "plausible_coverage_sbir", "plausible_coverage_sttr",
        "mean_sbir", "mean_sttr", "median_sbir", "median_sttr",
        "both_program_denominators", "count_supported", "coverage_supported",
        "supported", "mean_difference_days", "median_difference_days",
        "overlap_weight",
    ]
    if frame.empty:
        return pd.DataFrame(columns=columns)
    denominator_grouped = frame.groupby([*index, "program_n"], dropna=False)["award_key"]
    denominators = (
        denominator_grouped.size()
        .unstack("program_n")
        .reindex(columns=PROGRAMS, fill_value=0)
        .fillna(0)
        .astype(int)
    )
    plausible = frame.loc[frame["status"].eq("plausible")]
    plausible_grouped = plausible.groupby([*index, "program_n"], dropna=False)["days"]
    counts = (
        plausible_grouped.size()
        .unstack("program_n")
        .reindex(index=denominators.index, columns=PROGRAMS, fill_value=0)
        .fillna(0)
        .astype(int)
    )
    means = (
        plausible_grouped.mean().unstack("program_n").reindex(index=denominators.index, columns=PROGRAMS)
    )
    medians = (
        plausible_grouped.median().unstack("program_n").reindex(index=denominators.index, columns=PROGRAMS)
    )
    denominators.columns = [f"denominator_{program.lower()}" for program in denominators.columns]
    counts.columns = [f"plausible_n_{program.lower()}" for program in counts.columns]
    means.columns = [f"mean_{program.lower()}" for program in means.columns]
    medians.columns = [f"median_{program.lower()}" for program in medians.columns]
    out = pd.concat([denominators, counts, means, medians], axis=1).reset_index()
    for program in ("sbir", "sttr"):
        out[f"plausible_coverage_{program}"] = (
            out[f"plausible_n_{program}"] / out[f"denominator_{program}"]
        ).where(out[f"denominator_{program}"].gt(0))
    out["both_program_denominators"] = (
        out["denominator_sbir"].gt(0) & out["denominator_sttr"].gt(0)
    )
    out["count_supported"] = (
        out["plausible_n_sbir"].ge(MIN_PROGRAM_ROWS)
        & out["plausible_n_sttr"].ge(MIN_PROGRAM_ROWS)
    )
    out["coverage_supported"] = (
        out["count_supported"]
        & out["plausible_coverage_sbir"].ge(MIN_INTERVAL_COVERAGE)
        & out["plausible_coverage_sttr"].ge(MIN_INTERVAL_COVERAGE)
    )
    out["supported"] = out["coverage_supported"]
    out["mean_difference_days"] = out["mean_sttr"] - out["mean_sbir"]
    out["median_difference_days"] = out["median_sttr"] - out["median_sbir"]
    out["overlap_weight"] = out[["plausible_n_sbir", "plausible_n_sttr"]].min(axis=1)
    return out.loc[:, columns]


stratum_comparisons = build_stratum_comparisons(denominator_rows)
count_supported_strata = stratum_comparisons.loc[stratum_comparisons["count_supported"]].copy()
supported_strata = stratum_comparisons.loc[stratum_comparisons["supported"]].copy()
supported_strata.sort_values(["interval", "phase_n", "solicitation_year", "agency_n", "solicitation_n"])


In [ ]:
readiness_rows = []
for (interval, agency, phase), group in stratum_comparisons.groupby(
    ["interval", "agency_n", "phase_n"], dropna=False, sort=True
):
    count_supported = group.loc[group["count_supported"]]
    supported = group.loc[group["supported"]]
    readiness_rows.append(
        {
            "interval": interval,
            "agency": agency,
            "phase": phase,
            "all_observed_strata": len(group),
            "both_program_denominator_strata": int(group["both_program_denominators"].sum()),
            "count_supported_strata": len(count_supported),
            "coverage_supported_strata": len(supported),
            "supported_sbir_plausible": int(supported["plausible_n_sbir"].sum()),
            "supported_sttr_plausible": int(supported["plausible_n_sttr"].sum()),
            "supported_sbir_denominator": int(supported["denominator_sbir"].sum()),
            "supported_sttr_denominator": int(supported["denominator_sttr"].sum()),
        }
    )
readiness_map = pd.DataFrame(readiness_rows)
if readiness_map.empty:
    readiness_by_interval = pd.DataFrame()
else:
    readiness_by_interval = (
        readiness_map.groupby(["interval", "phase"], dropna=False)
        .agg(
            agencies=("agency", "nunique"),
            all_observed_strata=("all_observed_strata", "sum"),
            both_program_denominator_strata=("both_program_denominator_strata", "sum"),
            count_supported_strata=("count_supported_strata", "sum"),
            coverage_supported_strata=("coverage_supported_strata", "sum"),
            supported_sbir_plausible=("supported_sbir_plausible", "sum"),
            supported_sttr_plausible=("supported_sttr_plausible", "sum"),
            supported_sbir_denominator=("supported_sbir_denominator", "sum"),
            supported_sttr_denominator=("supported_sttr_denominator", "sum"),
        )
        .reset_index()
    )
    readiness_by_interval["supported_sbir_coverage"] = (
        readiness_by_interval["supported_sbir_plausible"]
        / readiness_by_interval["supported_sbir_denominator"]
    ).where(readiness_by_interval["supported_sbir_denominator"].gt(0))
    readiness_by_interval["supported_sttr_coverage"] = (
        readiness_by_interval["supported_sttr_plausible"]
        / readiness_by_interval["supported_sttr_denominator"]
    ).where(readiness_by_interval["supported_sttr_denominator"].gt(0))
threshold_rows = []
for threshold in COVERAGE_THRESHOLDS:
    threshold_label = (
        "count-only" if threshold == 0 else f"count + {threshold:.0%} coverage"
    )
    for (interval, phase), group in stratum_comparisons.groupby(
        ["interval", "phase_n"], dropna=False, sort=True
    ):
        selected = group.loc[
            group["count_supported"]
            & group["plausible_coverage_sbir"].ge(threshold)
            & group["plausible_coverage_sttr"].ge(threshold)
        ]
        threshold_rows.append(
            {
                "interval": interval,
                "phase": phase,
                "readiness_rule": threshold_label,
                "minimum_coverage": threshold,
                "supported_strata": len(selected),
                "sbir_plausible": int(selected["plausible_n_sbir"].sum()),
                "sttr_plausible": int(selected["plausible_n_sttr"].sum()),
                "sbir_denominator": int(selected["denominator_sbir"].sum()),
                "sttr_denominator": int(selected["denominator_sttr"].sum()),
            }
        )
threshold_sensitivity = pd.DataFrame(threshold_rows)
plausible_denominator_rows = denominator_rows.loc[denominator_rows["status"].eq("plausible")]
support_exclusions = pd.DataFrame(
    [
        {"stage": "all comparison source-key interval rows", "rows": len(comparison_universe)},
        {"stage": "stable complete-stratum denominator rows", "rows": len(denominator_rows)},
        {"stage": "plausible denominator rows", "rows": len(plausible_denominator_rows)},
        {"stage": "plausible rows in count-only supported strata", "rows": int(count_supported_strata[["plausible_n_sbir", "plausible_n_sttr"]].sum().sum()) if not count_supported_strata.empty else 0},
        {"stage": "plausible rows in 80%-coverage supported strata", "rows": int(supported_strata[["plausible_n_sbir", "plausible_n_sttr"]].sum().sum()) if not supported_strata.empty else 0},
    ]
)
if readiness_map.empty:
    supported_readiness = pd.DataFrame()
else:
    supported_readiness = readiness_map.loc[readiness_map["coverage_supported_strata"].gt(0)].sort_values(
        ["interval", "phase", "coverage_supported_strata", "agency"],
        ascending=[True, True, False, True],
    )
display(support_exclusions)
display(threshold_sensitivity)
display(readiness_by_interval)
supported_readiness


## Standardized contrasts — observed common support only

For each phase and interval, two fixed common-stratum targets are shown:

1. **Equal-stratum:** every supported stratum receives equal weight.
2. **Symmetric overlap:** a stratum receives `min(plausible_n_SBIR, plausible_n_STTR)` weight. Neither program supplies the target distribution.

Both average stratum-specific **means** after the 0–730-day gate. They are descriptive standardizations, not effects. The stratum table retains per-program denominators, coverage, and medians as diagnostics, but the notebook never pools source-key rows to produce a program median. The threshold-sensitivity table repeats the phase contrasts under count-only and count-plus-80%-coverage rules. A sign that changes across weighting rules, readiness thresholds, phases, agencies, intervals, or strata is a refusal signal, not something to average away.


In [ ]:
def standardized_contrasts(frame: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    rows = []
    if frame.empty:
        return pd.DataFrame()
    grouper = by[0] if len(by) == 1 else by
    for key, group in frame.groupby(grouper, dropna=False, sort=True):
        keys = (key,) if len(by) == 1 else key
        weights = group["overlap_weight"].astype(float)
        differences = group["mean_difference_days"].astype(float)
        record = dict(zip(by, keys, strict=True))
        record.update(
            {
                "target": "observed supported strata only",
                "n_strata": len(group),
                "plausible_n_sbir": int(group["plausible_n_sbir"].sum()),
                "plausible_n_sttr": int(group["plausible_n_sttr"].sum()),
                "denominator_sbir": int(group["denominator_sbir"].sum()),
                "denominator_sttr": int(group["denominator_sttr"].sum()),
                "plausible_coverage_sbir": (
                    float(group["plausible_n_sbir"].sum() / group["denominator_sbir"].sum())
                    if group["denominator_sbir"].sum() else pd.NA
                ),
                "plausible_coverage_sttr": (
                    float(group["plausible_n_sttr"].sum() / group["denominator_sttr"].sum())
                    if group["denominator_sttr"].sum() else pd.NA
                ),
                "equal_stratum_mean_difference_days": float(differences.mean()),
                "overlap_weighted_mean_difference_days": (
                    float((differences * weights).sum() / weights.sum()) if weights.sum() else pd.NA
                ),
                "strata_sttr_longer": int(differences.gt(0).sum()),
                "strata_equal": int(differences.eq(0).sum()),
                "strata_sttr_shorter": int(differences.lt(0).sum()),
                "min_stratum_difference_days": float(differences.min()),
                "max_stratum_difference_days": float(differences.max()),
            }
        )
        rows.append(record)
    return pd.DataFrame(rows)


threshold_standardized_frames = []
for threshold in COVERAGE_THRESHOLDS:
    selected = stratum_comparisons.loc[
        stratum_comparisons["count_supported"]
        & stratum_comparisons["plausible_coverage_sbir"].ge(threshold)
        & stratum_comparisons["plausible_coverage_sttr"].ge(threshold)
    ]
    standardized = standardized_contrasts(selected, ["interval", "phase_n"])
    if not standardized.empty:
        standardized.insert(0, "minimum_coverage", threshold)
        standardized.insert(0, "readiness_rule", (
            "count-only" if threshold == 0 else f"count + {threshold:.0%} coverage"
        ))
        threshold_standardized_frames.append(standardized)
threshold_standardized_by_phase = (
    pd.concat(threshold_standardized_frames, ignore_index=True)
    if threshold_standardized_frames else pd.DataFrame()
)
standardized_by_phase = standardized_contrasts(supported_strata, ["interval", "phase_n"])
standardized_by_agency_phase = standardized_contrasts(
    supported_strata, ["interval", "agency_n", "phase_n"]
)
display(threshold_standardized_by_phase)
display(standardized_by_phase)
standardized_by_agency_phase


## Conclusion / refusal

**Decision rule:** if no primary interval has strata meeting both the count and conservative 80% per-program coverage thresholds, refuse even the descriptive comparison. If supported strata exist, allow only a map of those observed record intervals and their sensitivity to endpoint, phase, agency, readiness threshold, and fixed common-stratum weights. In either case, refuse an administrative-friction estimate and a program-wide STTR-is-faster/slower statement.

The reason is structural, not merely low sample size: the export selects awardees, does not contain unsuccessful proposals, does not define a uniform submission/review clock, and does not show whether records sharing a solicitation identifier followed the same topic or review path.

**Reference-snapshot outcome (SHA-256 `1058456d…c600`):** the count-only rule admits 62 primary receipt-to-notification strata; the conservative 80% coverage rule admits 60 (30 per phase). Those 60 strata contain 4,953/4,964 plausible SBIR and 1,778/1,811 plausible STTR source-key intervals. Their symmetric overlap-weighted mean contrasts (STTR minus SBIR) are −2.8 days in Phase I and +8.0 days in Phase II; under count-only the Phase II contrast is +13.0 days. Stratum signs remain mixed within both phases. The lineage audit flags five plausible split-key groups spanning ten source keys and does not collapse them. The threshold sensitivity and within-phase spread support a bounded readiness map, not a single timing direction. These figures are exploratory and non-citable.


In [ ]:
primary = supported_strata.loc[
    supported_strata.get("interval", pd.Series(dtype=str)).eq("receipt_to_notification")
]
primary_count_only = count_supported_strata.loc[
    count_supported_strata.get("interval", pd.Series(dtype=str)).eq("receipt_to_notification")
]
if source_state == "missing":
    conclusion = (
        "REFUSAL — input absent. Supply a pinned SBIR.gov CSV with SBIR_AWARDS_CSV; "
        "missing local data must not become a zero-row finding."
    )
elif source_state == "malformed":
    conclusion = (
        "REFUSAL — input is present but malformed. Missing required identity, stratum, "
        "or endpoint columns: " + ", ".join(missing_required_columns) + "."
    )
elif source_state == "empty":
    conclusion = (
        "REFUSAL — input is present but has no analyzable source rows "
        "(zero-byte/no parseable columns or header-only); this is not a zero-award finding."
    )
elif primary.empty:
    conclusion = (
        "REFUSAL — no primary receipt-to-notification stratum meets both the count "
        f"and conservative {MIN_INTERVAL_COVERAGE:.0%} per-program coverage rule; "
        f"{len(primary_count_only):,} strata meet the count-only rule."
    )
else:
    conclusion = (
        f"DESCRIPTIVE-ONLY — {len(primary):,} agency×solicitation×phase×year strata "
        f"meet the count-plus-{MIN_INTERVAL_COVERAGE:.0%}-coverage receipt-to-notification rule, "
        f"versus {len(primary_count_only):,} under count-only. Supported strata contain "
        f"{int(primary['plausible_n_sbir'].sum()):,}/{int(primary['denominator_sbir'].sum()):,} "
        f"plausible SBIR and {int(primary['plausible_n_sttr'].sum()):,}/"
        f"{int(primary['denominator_sttr'].sum()):,} plausible STTR source-key intervals. Inspect the stratum, "
        "standardization tables; do not convert them into an administrative-friction or "
        "program-wide faster/slower finding."
    )
print(conclusion)
if not primary.empty:
    display(
        primary.groupby(["agency_n", "phase_n"], dropna=False)
        .agg(
            supported_strata=("solicitation_n", "size"),
            sbir_plausible=("plausible_n_sbir", "sum"),
            sttr_plausible=("plausible_n_sttr", "sum"),
            sbir_denominator=("denominator_sbir", "sum"),
            sttr_denominator=("denominator_sttr", "sum"),
        )
        .reset_index()
        .assign(
            sbir_coverage=lambda x: x["sbir_plausible"] / x["sbir_denominator"],
            sttr_coverage=lambda x: x["sttr_plausible"] / x["sttr_denominator"],
        )
        .sort_values(["phase_n", "supported_strata", "agency_n"], ascending=[True, False, True])
    )


## Interpretation limits

- **Awardee-only selection:** these rows condition on receiving an award. They omit rejected and withdrawn proposals. A difference among winners can reflect selection, not processing time for applicants.
- **Source-key grain is not a verified physical award:** each retained row is one `sbir-source-v2` key after repeated editions of that key are audited. Plausible identifier-correction lineages can split one lineage across keys; they are reported but not collapsed without a governed rule.
- **Unobserved clocks:** the export does not expose initial submission, completeness checks, resubmissions, reviewer assignment, panel, score, selection, negotiation, obligation, or a verified contract start. Source labels are not a uniform workflow definition.
- **Shared solicitation is not matched treatment:** one identifier can span topics, deadlines, agencies' internal tracks, review panels, or program-specific rules. Topic Code coverage is audited but not assumed to solve this.
- **Residual mix:** firm characteristics, research-institution coordination, agency policy, topic complexity, budget, and calendar can differ inside the declared stratum.
- **No causal inference:** program participation is not randomized. The standardized contrast changes the observed stratum weights; it does not identify a causal STTR effect.
- **No burden or cost measure:** calendar days are not applicant hours, reviewer hours, agency dollars, or cost per awarded dollar. C4's allocation-cost study remains the relevant cost framework.
- **No pooled ranking:** pooled program medians would reintroduce agency, phase, year, and solicitation composition. This notebook does not compute them.
- **Gate and readiness sensitivity:** the 0–730-day rule prevents obvious reversals and extreme spans from driving comparisons. It can also exclude real long cycles; excluded counts stay visible. The 80% coverage threshold is a conservative readiness convention, not statistical truth, so count-only and 80% results are both reported.

**Bottom line:** public records can reveal where *recorded* timing is sufficiently populated for a within-observed-stratum diagnostic. They cannot establish that STTR has more or less administrative friction than SBIR.


## Promotion checklist

- [x] Question and decision are explicit.
- [x] Data snapshot rule, population, grain, keys, exclusions, assumptions, and seed are recorded.
- [x] Missing inputs degrade to a refusal, and present-but-malformed inputs fail schema validation.
- [x] SBIR/STTR and Phase I/II remain separate until fixed common-stratum standardization.
- [x] Missing, implausible, and interval-specific edition conflicts remain visible.
- [x] Every stratum reports program denominators and plausible-interval coverage; count-only and 80% rules are compared.
- [x] Split-key lineage candidates are audited and not collapsed.
- [x] The destination tier is explicit (`exploratory`, non-citable).
- [ ] Recurring calculations have one canonical implementation.
- [ ] Recurring artifact generation has a thin CLI or Dagster asset.
- [ ] Citable work satisfies all four `evidence` contract requirements.
- [x] Outputs and execution counts are cleared before commit.

Until an explicit promotion supplies defined workflow events, applicant denominators, and an identification design, this notebook and its record-interval contrasts remain exploratory and non-citable.
